In [1]:
# import napari and numpy
import napari
import numpy as np
import tifffile
import zarr
import dask.array
import pandas as pd
import os

In [7]:
crop_dir = "../../../../../../../../sds_mount/sds/sd22c003/phenotyping_benchmark/visual_qc/crop_data/"
dataset_dir = "../../../../../../../../sds_mount/sds/sd22c003/phenotyping_benchmark/datasets/"

In [9]:
dataset = 'IMMUcan'
sample = 'IMMUcan_2022_WFLOW_10034294-GI-VAR-TIS-UNST-03_004'
crop_number = '0001'

In [4]:
data = pd.read_csv(os.path.join(dataset_dir, f'{dataset}/quantification/processed/{dataset}_quantification.csv'))

In [22]:

cell_data = data[(data['sample_id'] == sample)].copy() 
cell_data['annotator1'] = 'NA'
cell_data['annotator2'] = 'NA'

In [23]:

with open(os.path.join(dataset_dir, f'{dataset}/markers.txt')) as f:
    markers = [line.strip() for line in f.readlines()]
segmentation_mask_path = os.path.join(crop_dir, f'{dataset}/crop_{sample}_{crop_number}_mask.tiff')
image_path = os.path.join(crop_dir, f'{dataset}/crop_{sample}_{crop_number}.ome.tiff')
image = tifffile.imread(image_path)


n_channels = len(markers)
colormap = ['cyan', 'green', 'yellow', 'red', 'magenta', 'orange', 'blue']
colormap = colormap * (len(markers) // len(colormap) + 1)

viewer = napari.Viewer()
for i in range(n_channels):
    channel_data = image[i]  # Extract the i-th channel
    viewer.add_image(
        channel_data,
        name=markers[i],
        rgb=False,
        contrast_limits=[0, 65535],
        blending='additive',
        colormap=colormap[i],
    )



segmentation_mask = tifffile.imread(segmentation_mask_path)
viewer.add_labels([segmentation_mask, segmentation_mask[::2,::2], segmentation_mask[::4,::4]], name='segmentation')
for i, cluster in enumerate(cell_data['cell_type'].unique()):
    layer_labels = cell_data.loc[cell_data['cell_type'] == cluster, 'cell_id']
    layer_features = cell_data.loc[cell_data['cell_type'] == cluster, ['cell_id', 'cell_type', 'annotator1', 'annotator2']]
    segmentation_mask_binary = np.zeros(segmentation_mask.shape, dtype=np.uint8)
    segmentation_mask_binary[np.isin(segmentation_mask, layer_labels)] = i
    viewer.add_labels([segmentation_mask_binary, segmentation_mask_binary[::2,::2], segmentation_mask_binary[::4,::4]], name=cluster, features=layer_features)

In [31]:
cell_data = cell_data.set_index('cell_id')
for layer in viewer.layers:
    if hasattr(layer, 'features') and 'cell_type' in layer.features.columns:
        edited = layer.features[['cell_id', 'annotator1', 'annotator2']].set_index('cell_id')
        cell_data.update(edited)
cell_data = cell_data.reset_index()

In [ ]:
cell_data.to_csv(os.path.join(f'../../../../../../../../sds_mount/sds/sd22c003/phenotyping_benchmark/visual_qc/corrected_annotations/{dataset}_quantification_annotator1.csv'), index=False)